# Analysis of correlations

## Import data and define functions for analysis

In [ ]:
import pandas as pd
import numpy as np
import datetime as DT
import requests
import json 
import warnings
import seaborn as sns
from sklearn.cluster import KMeans  # to cluster

from linearmodels.panel import PanelOLS

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.dates as mdates
from scipy import stats as sps
warnings.filterwarnings('ignore')
from adjustText import adjust_text
from datetime import date

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

df=pd.read_csv('FULL_DATA5_FINALE.csv')
indicators = [
    "mask",
    "worried_catch_covid",
    "belief_masking_effective",
    "received_news_local_health",
    "received_news_experts",
    "received_news_who",
    "received_news_govt_health",
    "received_news_politicians",
    "received_news_journalists",
    "received_news_friends",
    "trust_covid_info_local_health",
    "trust_covid_info_experts",
    "trust_covid_info_who",
    "trust_covid_info_govt_health",
    "trust_covid_info_politicians",
    "trust_covid_info_journalists",
    "trust_covid_info_friends"
]

In [ ]:
import scipy.stats # shannon entropy
def shannon_entropy_scipy(data):   
    return scipy.stats.entropy(data, base=2)

def shannon(countri, ind): # shannon entropy H
    dfj = df[df['country'] == countri].copy()
    counts, bin_edges = np.histogram(dfj[ind].dropna(), bins='fd')  # automatic bin subdivision (fd - Freedman-Diaconis)
    probabilities = counts / (counts.sum())
   
    nonzero_probs = probabilities[probabilities > 0]
    H = shannon_entropy_scipy(nonzero_probs)
    H_normalized = H / np.log2(len(nonzero_probs)) if len(nonzero_probs) > 1 else 0  # Normalization
    
    return H


In [ ]:
def calculate_correlations(var1, var2):
    """calculate correlations between two variables, for each country"""
    corr_list = []
    for country, group in df.groupby('country'):
        group_clean = group[[var1, var2]].dropna()
        if len(group_clean) >= 2:
            corr = group_clean.corr().iloc[0, 1]
            corr_list.append({'country': country, 'correlation': corr})
    
    return pd.DataFrame(corr_list)

In [ ]:
# Add outliers to boxplot
def add_outliers(ax, data, x_pos, color):
    label_offset = 0.02
    min_spacing = 0.05  # dmin dist among labels
    
    Q1 = data['correlation'].quantile(0.25)
    Q3 = data['correlation'].quantile(0.75)
    IQR = Q3 - Q1
    outliers = data[(data['correlation'] < (Q1 - 1.5*IQR)) |  (data['correlation'] > (Q3 + 1.5*IQR))]
    
    # order outliers according to corr
    outliers = outliers.sort_values('correlation')
    previous_y = None
    
    for _, row in outliers.iterrows():
        current_y = row['correlation']
        
        # regulate y position if too close to former
        if previous_y is not None and abs(current_y - previous_y) < min_spacing:
            current_y = previous_y + min_spacing * (1 if current_y > previous_y else -1)
        
        ax.plot(x_pos, row['correlation'], 'o', 
                color=color,
                markersize=6,
                markeredgewidth=0.7,
                markeredgecolor=color)

        ax.text(x_pos + label_offset,
                current_y,
                f" {row['country']}",
                ha='left', 
                va='center',
                color=color,
                fontsize=10)
        
        previous_y = current_y

In [ ]:
# Granger causality

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import grangercausalitytests

maxlag = 28 # max time lag considered
test   = 'ssr_chi2test'
def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False):    
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables) # instantiate matrix
    for c in df.columns: # for each variable x
        for r in df.index: # for each variable y
            test_result = grangercausalitytests(data[[r, c]], maxlag=maxlag, verbose=False) # test
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(maxlag)]   # compute pvalue for each lag
            if verbose: print(f'Y = {r}, X = {c}, P Values = {p_values}')
            min_p_value = np.min(p_values) # min pvalue over lags
            df.loc[r, c] = min_p_value # save in final matrix
    df.columns = [var + '_x' for var in variables] # rename matrix labels
    df.index = [var + '_y' for var in variables]
    return df

In [ ]:
def shannon(countri, ind):
    dfj = df[df['country'] == countri].copy()
    counts, bin_edges = np.histogram(dfj[ind].dropna(), bins='fd')
    probabilities = counts / counts.sum()
    
    H = scipy.stats.entropy(probabilities, base=2)
    max_entropy = np.log2(len(probabilities))  # log2 del numero totale di bin
    
    H_normalized = H / max_entropy if max_entropy > 0 else 0
    return H_normalized

In [ ]:
def boxplot1(correlations_df1, name1, title, nomesalvataggio):
    """Immagine con 1 boxplot"""
    # Prepare data
    df1 = correlations_df1.copy()
    df1['dataset'] = name1
    combined_df = df1
    #figura
    plt.figure(figsize=(6, 6))
    ax = sns.boxplot(
        x='dataset',
        y='correlation',
        data=combined_df,
        width=0.3,
        showfliers=False,
        boxprops=dict(facecolor='white', edgecolor='black', linewidth=1), #per fare grandezza dei plot
        whiskerprops=dict(color='black', linewidth=1),
        capprops=dict(color='black', linewidth=1),
        medianprops=dict(color='red', linewidth=2)
    )

    # Add outlier
    add_outliers(ax, df1, 0)
    ax.set_xticklabels([name1], fontsize=14)

    # Format
    plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
    plt.ylim(-1, 1)
    plt.title(title, fontsize=14, pad=15)
    plt.ylabel('Correlations', fontsize=14)
    plt.xlabel('',fontsize=14)
    plt.grid(axis='y', linestyle=':', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
def boxplot2(correlations_df1, correlations_df2, name1, name2, title, nomesalvataggio, color):
    """Immagine con 2 boxplot"""
    # Prepare data
    df1 = correlations_df1.copy()
    df2 = correlations_df2.copy()
    df1['dataset'] = name1
    df2['dataset'] = name2
    combined_df = pd.concat([df1, df2])

    #fig
    plt.figure(figsize=(4, 6.5))
    ax = sns.boxplot(
        x='dataset',
        y='correlation',
        data=combined_df,
        width=0.5,
        showfliers=False,
        boxprops=dict(facecolor='white', edgecolor=color, linewidth=0.7), 
        whiskerprops=dict(color=color, linewidth=0.7),
        capprops=dict(color=color, linewidth=0.7),
        medianprops=dict(color=color, linewidth=1)
    )

    # Add outlier
    add_outliers(ax, df1, 0, color)
    add_outliers(ax, df2, 1, color)
    current_xlim = ax.get_xlim()
    ax.set_xlim(current_xlim[0], current_xlim[1] + 0.4) # add space for labels
    ax.set_xticklabels([name1, name2], fontsize=12)
    ax.tick_params(axis='x', labelrotation=15)
    
    plt.axhline(0, color='gray', linestyle=':', alpha=0.4)
    plt.title(title, fontsize=14, pad=15, color=color)
    plt.ylabel('Correlations $\\rho_{XY}$', fontsize=14)
    plt.xlabel('',fontsize=14)
    plt.grid(axis='y', linestyle=':', alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
def boxplot7(correlations_dfs, names, title, nomesalvataggio, color):
    """ 7 boxplot nell'immagine"""

    combined_df = pd.DataFrame()
    for i in range(len(correlations_dfs)): #combino i dataset
        dfi = correlations_dfs[i].copy()
        dfi['dataset'] = names[i]
        combined_df = pd.concat([combined_df, dfi])

    #figure
    plt.figure(figsize=(7, 6.5))
    ax = sns.boxplot(
        x='dataset',
        y='correlation',
        data=combined_df,
        width=0.5,
        showfliers=False,
        boxprops=dict(facecolor='white', edgecolor=color, linewidth=0.7),
        whiskerprops=dict(color=color, linewidth=0.7),
        capprops=dict(color=color, linewidth=0.7),
        medianprops=dict(color=color, linewidth=1)
    )

    for i in range(len(correlations_dfs)):     # aggiungo gli outlier per ogni dataset
        dfi = correlations_dfs[i].copy()
        dfi['dataset'] = names[i]
        add_outliers(ax, dfi, i, color)

    current_xlim = ax.get_xlim()
    ax.set_xlim(current_xlim[0], current_xlim[1] + 0.5) #per aggiungere spazio a dx per le etichette

    plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
    plt.title(title, fontsize=15, pad=15, color=color)
    plt.ylabel('', fontsize=15)
    plt.xlabel('')
    plt.xticks(fontsize=14)
    ax.tick_params(axis='x', labelrotation=15, color=color)
    plt.grid(axis='y', linestyle=':', alpha=0.3)
    plt.show()

In [ ]:
def plot_aree(corr, title, nomesalvataggio): # plot forgeagraphhical areas
    countries = df['country'].unique()
    dl= corr
    dl['Area'] = dl['country'].apply(continent)
    dl_sorted = dl.sort_values(by='correlation', ascending=False)
    kmeans = KMeans(n_clusters=3, random_state=0)
    kmeans.fit(dl[['correlation']])  # fit sul dato 2D
    centroids = kmeans.cluster_centers_.flatten() # Kmeans centroids
    centroids.sort()

    # Division lines
    division_lines=[]
    for i in range(len(centroids)-1):
        division_lines.append((centroids[i] + centroids[i+1])/2)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.scatterplot(x='correlation', y='country', data=dl_sorted, s=100, hue='Area',palette=palettes, ax=ax)
    for line in division_lines:
        ax.axvline(x=line, color='red', linestyle='--') #d Divide clusters
    ax.set_title(title)
    ax.grid(True)
    #ax.legend(title='Geographic Area', bbox_to_anchor=(1.05, 1), loc='upper left')

    leg1 = ax.legend(title='Geographic Area', bbox_to_anchor=(1.05, 1), loc='upper left')
    red_line = Line2D([0], [0], color='red', linestyle='--', label='Cluster division K-Means')
    leg2 = ax.legend(handles=[red_line], bbox_to_anchor=(1.05, 0.70), loc='upper left', frameon=True)
    ax.add_artist(leg1)
    plt.tight_layout()

## Beliefs

### Box plots

In [ ]:
corr1=calculate_correlations("pct_mask","pct_worried_catch_covid")
corr2=calculate_correlations("pct_mask", "pct_belief_masking_effective")
boxplot2(corr1,corr2, 'Worried catch COVID', 'Masking effective', 'Beliefs', 'CorrBeliefs', 'black')

### Test on variability using Shannon entropy

In [ ]:
print(shannon('Japan', 'pct_mask')) #beh
print(shannon('Thailand', 'pct_mask'))
print(shannon('Malaysia', 'pct_mask'))
print(shannon('New Zealand', 'pct_mask'))

print(shannon('Japan', 'pct_worried_catch_covid')) #bel1
print(shannon('Thailand', 'pct_worried_catch_covid'))
print(shannon('Malaysia', 'pct_worried_catch_covid'))

print(shannon('New Zealand', 'pct_belief_masking_effective')) #bel2
print(shannon('Malaysia', 'pct_belief_masking_effective'))

### Granger causality

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_worried_catch_covid']].dropna(), variables =['pct_mask', 'pct_worried_catch_covid']) 
#in quetso caso ho che <0.05 ottengo che x do not granger cause y.  invece altro ho che new deaths è granger cause di y

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_belief_masking_effective']].dropna(), variables =['pct_mask', 'pct_belief_masking_effective']) 

### MLR

In [ ]:
# This groups all countries together and does not separate the possibility of having within-country autocorrelation or cross-country correlation

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

b = df[['pct_mask', 'pct_worried_catch_covid', 'pct_belief_masking_effective']].copy().dropna()

X = b[['pct_worried_catch_covid', 'pct_belief_masking_effective']]
y = b['pct_mask']

X = sm.add_constant(X)  # aggiunge la colonna 'const'

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
# VIF: helps to understand how much the independence assumption holds
X['intercept'] = 1
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

In [ ]:
# This additional analysis allows to disentangle the pooled estimate and isolates the within-country association 
# by removing the removing the country-specific average as well as the temporal within-country and corss-country correlations

# Panel-indexed DataFrame (sorted by country)
b = df[['country', 'survey_date', 'pct_mask', 'pct_worried_catch_covid', 'pct_belief_masking_effective']].copy()
b = b.dropna()
b['survey_date'] = pd.to_datetime(b['survey_date'])
b = b.set_index(['country', 'survey_date'])

y = b['pct_mask']
X = b[['pct_worried_catch_covid', 'pct_belief_masking_effective']]

# Country fixed-effects ("within") estimator; entity_effects=True implements the demeaning
model_fe = PanelOLS(y, X, entity_effects=True)

# Cluster-robust SEs by country: robust to within-country autocorrelation of arbitrary form
fe_clustered = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(fe_clustered.summary)

# Driscoll-Kraay SEs: robust to within-country autocorrelation AND cross-country contemporaneous correlation (e.g. shared global shocks/news events). 
# Doesn't assume that countries are independent and includes heteroskedasticity
fe_dk = model_fe.fit(cov_type='kernel', kernel='bartlett', bandwidth=12)
print(fe_dk.summary)


## Awareness

In [ ]:
corr1=calculate_correlations("pct_mask","pct_received_news_local_health")
corr2=calculate_correlations("pct_mask","pct_received_news_experts")
corr3=calculate_correlations("pct_mask","pct_received_news_who")
corr4=calculate_correlations("pct_mask","pct_received_news_govt_health")
corr5=calculate_correlations("pct_mask","pct_received_news_politicians")
corr6=calculate_correlations("pct_mask","pct_received_news_journalists")
corr7=calculate_correlations("pct_mask","pct_received_news_friends")
corr=[corr1, corr2, corr3, corr4, corr5, corr6, corr7]
names=['Awa1', 'Awa2','Awa3','Awa4','Awa5','Awa6','Awa7']
boxplot7(corr, names, 'Awareness', 'BA', 'darkorchid')

### Understand awareness for all sources among countries

clusterss-> kemnas-> create clusters using KMeans 

In [ ]:
Europe= ['Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France', 'Germany',
               'Greece', 'Hungary', 'Italy', 'Netherlands', 'Norway', 'Poland', 'Portugal',
               'Romania', 'Spain', 'Sweden', 'Switzerland', 'Turkey', 'United Kingdom']  
Latin_America=['Argentina', 'Chile', 'Colombia', 'Ecuador', 'Peru', 'Mexico']   
Asia=['Indonesia', 'Japan', 'Malaysia', 'Philippines', 'Thailand'] 
Oceania=['Australia', 'New Zealand']
North_America= ['Canada']

def continent(country):
    if country in Europe:
        return 'Europe'
    elif country in Latin_America:
        return 'Latin_America'
    elif country in Asia:
        return 'Asia'
    elif country in Oceania:
        return 'Oceania'
    elif country in North_America:
        return 'North_America'
    else:
        return 'Other'
    
palettes = {
    'Europe':        '#0173B2',  # blu
    'Asia':          '#CC78BC',  # viola
    'Latin_America': '#DE8F05',  # arancione
    'North_America': '#D55E00',  # rosso-mattone
    'Oceania':       '#029E73'   # verde
}



In [ ]:
plot_aree(corr5, 'Country correlation of Behaviour and Awareness 5 (Politicians)', 'BAanalisi5')
plot_aree(corr6, 'Country correlation of Behaviour and Awareness 6 (Journalists)', 'BAanalisi6')

### MLR

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

b = df[['pct_mask', 'pct_received_news_local_health', 'pct_received_news_experts', 'pct_received_news_who', 'pct_received_news_govt_health', 'pct_received_news_politicians', 'pct_received_news_journalists', 'pct_received_news_friends' ]].copy().dropna()

X = b[['pct_received_news_local_health', 'pct_received_news_experts', 'pct_received_news_who', 'pct_received_news_govt_health', 'pct_received_news_politicians', 'pct_received_news_journalists', 'pct_received_news_friends' ]]
y = b['pct_mask']

X = sm.add_constant(X)  # aggiunge la colonna 'const'

model = sm.OLS(y, X).fit()

print(model.summary())


In [ ]:
# VIF
X['intercept'] = 1
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

In [ ]:
# This additional analysis allows to disentangle the pooled estimate and isolates the within-country association 
# by removing the removing the country-specific average as well as the temporal within-country and corss-country correlations

# Panel-indexed DataFrame (sorted by country)
b = df[['country', 'survey_date', 'pct_mask',  'pct_received_news_local_health', 'pct_received_news_experts', 'pct_received_news_who', 'pct_received_news_govt_health', 'pct_received_news_politicians', 'pct_received_news_journalists', 'pct_received_news_friends']].copy()
b = b.dropna()
b['survey_date'] = pd.to_datetime(b['survey_date'])
b = b.set_index(['country', 'survey_date'])

y = b['pct_mask']
X = b[[ 'pct_received_news_local_health', 'pct_received_news_experts', 'pct_received_news_who', 'pct_received_news_govt_health', 'pct_received_news_politicians', 'pct_received_news_journalists', 'pct_received_news_friends' ]]

# Country fixed-effects ("within") estimator; entity_effects=True implements the demeaning
model_fe = PanelOLS(y, X, entity_effects=True)

# Cluster-robust SEs by country: robust to within-country autocorrelation of arbitrary form
fe_clustered = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(fe_clustered.summary)

# Driscoll-Kraay SEs: robust to within-country autocorrelation AND cross-country contemporaneous correlation (e.g. shared global shocks/news events). 
# Doesn't assume that countries are independent and includes heteroskedasticity
fe_dk = model_fe.fit(cov_type='kernel', kernel='bartlett', bandwidth=12)
print(fe_dk.summary)

### Granger causality

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_received_news_local_health']].dropna(), variables =['pct_mask', 'pct_received_news_local_health']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_received_news_experts']].dropna(), variables =['pct_mask', 'pct_received_news_experts']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_received_news_who']].dropna(), variables =['pct_mask', 'pct_received_news_who']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_received_news_govt_health']].dropna(), variables =['pct_mask', 'pct_received_news_govt_health']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_received_news_politicians']].dropna(), variables =['pct_mask', 'pct_received_news_politicians']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_received_news_journalists']].dropna(), variables =['pct_mask', 'pct_received_news_journalists']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_received_news_friends']].dropna(), variables =['pct_mask', 'pct_received_news_friends']) 

## Trust

In [ ]:
cort1=calculate_correlations("pct_mask","pct_trust_covid_info_local_health")
cort2=calculate_correlations("pct_mask","pct_trust_covid_info_experts")
cort3=calculate_correlations("pct_mask","pct_trust_covid_info_who")
cort4=calculate_correlations("pct_mask","pct_trust_covid_info_govt_health")
cort5=calculate_correlations("pct_mask","pct_trust_covid_info_politicians")
cort6=calculate_correlations("pct_mask","pct_trust_covid_info_journalists")
cort7=calculate_correlations("pct_mask","pct_trust_covid_info_friends")
cort=[cort1, cort2, cort3, cort4, cort5, cort6, cort7]
names=['Tru1', 'Tru2','Tru3','Tru4','Tru5','Tru6','Tru7']
boxplot7(cort, names, 'Trust', 'BT', 'tab:blue')

### MLR

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

b = df[['pct_mask', 'pct_trust_covid_info_local_health', 'pct_trust_covid_info_experts', 'pct_trust_covid_info_who', 'pct_trust_covid_info_govt_health', 'pct_trust_covid_info_politicians', 'pct_trust_covid_info_journalists', 'pct_trust_covid_info_friends' ]].copy().dropna()

X = b[['pct_trust_covid_info_local_health', 'pct_trust_covid_info_experts', 'pct_trust_covid_info_who', 'pct_trust_covid_info_govt_health', 'pct_trust_covid_info_politicians', 'pct_trust_covid_info_journalists', 'pct_trust_covid_info_friends' ]]
y = b['pct_mask']

X = sm.add_constant(X)  # aggiunge la colonna 'const'

model = sm.OLS(y, X).fit()

print(model.summary())


In [ ]:
# VIF
X['intercept'] = 1
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

In [ ]:
# This additional analysis allows to disentangle the pooled estimate and isolates the within-country association 
# by removing the removing the country-specific average as well as the temporal within-country and corss-country correlations

# Panel-indexed DataFrame (sorted by country)
b = df[['country', 'survey_date', 'pct_mask',  'pct_trust_covid_info_local_health', 'pct_trust_covid_info_experts', 'pct_trust_covid_info_who', 'pct_trust_covid_info_govt_health', 'pct_trust_covid_info_politicians', 'pct_trust_covid_info_journalists', 'pct_trust_covid_info_friends' ]].copy()
b = b.dropna()
b['survey_date'] = pd.to_datetime(b['survey_date'])
b = b.set_index(['country', 'survey_date'])

y = b['pct_mask']
X = b[[ 'pct_trust_covid_info_local_health', 'pct_trust_covid_info_experts', 'pct_trust_covid_info_who', 'pct_trust_covid_info_govt_health', 'pct_trust_covid_info_politicians', 'pct_trust_covid_info_journalists', 'pct_trust_covid_info_friends'  ]]

# Country fixed-effects ("within") estimator; entity_effects=True implements the demeaning
model_fe = PanelOLS(y, X, entity_effects=True)

# Cluster-robust SEs by country: robust to within-country autocorrelation of arbitrary form
fe_clustered = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(fe_clustered.summary)

# Driscoll-Kraay SEs: robust to within-country autocorrelation AND cross-country contemporaneous correlation (e.g. shared global shocks/news events). 
# Doesn't assume that countries are independent and includes heteroskedasticity
fe_dk = model_fe.fit(cov_type='kernel', kernel='bartlett', bandwidth=12)
print(fe_dk.summary)

### Granger causality

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_trust_covid_info_local_health']].dropna(), variables =['pct_mask', 'pct_trust_covid_info_local_health']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_trust_covid_info_experts']].dropna(), variables =['pct_mask', 'pct_trust_covid_info_experts']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_trust_covid_info_who']].dropna(), variables =['pct_mask', 'pct_trust_covid_info_who']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_trust_covid_info_politicians']].dropna(), variables =['pct_mask', 'pct_trust_covid_info_politicians']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_trust_covid_info_journalists']].dropna(), variables =['pct_mask', 'pct_trust_covid_info_journalists']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_trust_covid_info_friends']].dropna(), variables =['pct_mask', 'pct_trust_covid_info_friends']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'pct_trust_covid_info_govt_health']].dropna(), variables =['pct_mask', 'pct_trust_covid_info_govt_health']) 

## Awareness/trust

In [ ]:
chi = ['local_health','experts','who',  'govt_health', 'politicians', 'journalists', 'friends' ]

for src in chi:
    awareness_col = f'pct_received_news_{src}'
    trust_col = f'pct_trust_covid_info_{src}'
    new_col = f'at_{src}'
    
    # crea la nuova colonna awareness / trust
    df[new_col] = df[awareness_col] / df[trust_col]


In [ ]:
cor1=calculate_correlations("pct_mask","at_local_health")
cor2=calculate_correlations("pct_mask","at_experts")
cor3=calculate_correlations("pct_mask","at_who")
cor4=calculate_correlations("pct_mask","at_govt_health")
cor5=calculate_correlations("pct_mask","at_politicians")
cor6=calculate_correlations("pct_mask","at_journalists")
cor7=calculate_correlations("pct_mask","at_friends")

cor=[cor1, cor2, cor3, cor4, cor5, cor6, cor7]
names=['Awa/Tru1', 'Awa/Tru2', 'Awa/Tru3', 'Awa/Tru4', 'Awa/Tru5', 'Awa/Tru6', 'Awa/Tru7']
boxplot7(cor, names, 'Correlation Between Behavior and Awareness/Trust Ratio', 'BAT',"black")


In [ ]:
plot_aree(cor6, 'Country correlation of Behaviour and Awareness/Trust 6 (Journalists)', 'BATanalisi6')

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'at_local_health']].dropna(), variables =['pct_mask', 'at_local_health']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'at_experts']].dropna(), variables =['pct_mask', 'at_experts']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'at_who']].dropna(), variables =['pct_mask', 'at_who']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'at_govt_health']].dropna(), variables =['pct_mask', 'at_govt_health']) 

In [ ]:
df_clean = df[['pct_mask', 'at_politicians']].copy()
df_clean = df_clean.dropna()
df_clean = df_clean[~df_clean.isin([np.inf, -np.inf]).any(axis=1)]
grangers_causation_matrix(df_clean , variables =['pct_mask', 'at_politicians']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'at_journalists']].dropna(), variables =['pct_mask', 'at_journalists']) 

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'at_friends']].dropna(), variables =['pct_mask', 'at_friends']) 

## New deaths VS behaviour

In [ ]:
df['new_deaths_smoothed'].max()

In [ ]:
df['new_deaths_smoothed'] = df['new_deaths_smoothed']/df['new_deaths_smoothed'].max()

In [ ]:
behdeath = df[['country', 'pct_mask', 'survey_date', 'total_deaths', 'new_deaths_smoothed']].copy()
behdeath['survey_date'] = pd.to_datetime(behdeath['survey_date'])

lista_paesi = behdeath['country'].unique()  
fig, axs = plt.subplots(12, 3, figsize=(18, 50), sharex=False)
axs = axs.flatten() 

for i, paese in enumerate(lista_paesi):
    df_paese = behdeath[behdeath['country'] == paese].copy()
    df_paese.set_index('survey_date', inplace=True)     # datetime index
    # mean over 7 days
    pct_mask_smooth = df_paese['pct_mask'].rolling('7D', min_periods=1).mean()
    df_smooth = df_paese.copy()
    df_smooth['pct_mask'] = pct_mask_smooth
    # cambia indice per il plot
    df_smooth.reset_index(inplace=True)

    ax1 = axs[i]
    ax1.plot(df_smooth['survey_date'], df_smooth['pct_mask'], color='blue', label='Mask Wearing Behaviour (%)')
    ax1.set_ylabel('Mask Wearing Behaviour (%)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    ax2 = ax1.twinx()
    ax2.plot(df_smooth['survey_date'], df_smooth['new_deaths_smoothed'], color='red', label='New Deaths')
    ax2.set_ylabel('New Deaths', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    ax1.set_title(paese)
    ax1.set_xlabel('Survey Date')

    #date asse x non tutte ma solo alcune 
    ax1.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

# Rimuove subplot vuoti
for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()


### Granger causality

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'new_deaths_smoothed']].dropna(), variables =['pct_mask', 'new_deaths_smoothed']) 
#in quetso caso ho che <0.05 ottengo che x do not granger cause y.  invece altro ho che new deaths è granger cause di y

Note: there are two spurious peaks in Ecuador and Chile, likely due to data reporting errors: they can be smoothed for the analysis

In [ ]:
dfft=df[df['country']== 'Chile'][['survey_date', 'new_deaths_smoothed']]
dfft=dfft[(dfft['survey_date']> '2022-03-20') & (dfft['survey_date']< '2022-04-1')]
plt.figure(figsize=(10, 5))
plt.plot(dfft['survey_date'], dfft['new_deaths_smoothed'], color='crimson', linewidth=2)

plt.title('New Deaths Smoothed in Chile ', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('New Deaths Smoothed', fontsize=12)
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
dfft.to_csv('try.csv')  #1694.0
df.loc[(df['country'] == 'Chile') & (df['new_deaths_smoothed'] > 1500), 'new_deaths_smoothed'] = 74.714

# 93.571 before peak
# 55.857 after peak
# mean= 74.714

In [ ]:

dfft=df[df['country']== 'Ecuador'][['survey_date', 'new_deaths_smoothed']]
dfft=dfft[(dfft['survey_date']> '2021-07-01') & (dfft['survey_date']< '2021-09-01')]
plt.figure(figsize=(10, 5))
plt.plot(dfft['survey_date'], dfft['new_deaths_smoothed'], color='crimson', linewidth=2)

plt.title('New Deaths Smoothed in Ecuador', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('New Deaths Smoothed', fontsize=12)
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
dfft.to_csv('try.csv') #1266.286
df.loc[(df['country'] == 'Ecuador') & (df['new_deaths_smoothed'] > 1200), 'new_deaths_smoothed'] = 68
# before peak 17.0
# after peak 119.143
# mean  68

In [ ]:
behdeath = df[['country', 'pct_mask', 'survey_date', 'total_deaths', 'new_deaths_smoothed']].copy()
behdeath['survey_date'] = pd.to_datetime(behdeath['survey_date'])

lista_paesi = behdeath['country'].unique()  
fig, axs = plt.subplots(12, 3, figsize=(18, 50), sharex=False)
axs = axs.flatten() 

for i, paese in enumerate(lista_paesi):
    df_paese = behdeath[behdeath['country'] == paese].copy()
    df_paese.set_index('survey_date', inplace=True)     # datetime index
    # Mean over 7 days
    pct_mask_smooth = df_paese['pct_mask'].rolling('7D', min_periods=1).mean()
    df_smooth = df_paese.copy()
    df_smooth['pct_mask'] = pct_mask_smooth
    # cambia indice per il plot
    df_smooth.reset_index(inplace=True)

    ax1 = axs[i]
    ax1.plot(df_smooth['survey_date'], df_smooth['pct_mask'], color='blue', label='Mask Wearing Behaviour (%)')
    ax1.set_ylabel('Mask Wearing Behaviour (%)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    ax2 = ax1.twinx()
    ax2.plot(df_smooth['survey_date'], df_smooth['new_deaths_smoothed'], color='red', label='New Deaths')
    ax2.set_ylabel('New Deaths', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    ax1.set_title(paese)
    ax1.set_xlabel('Survey Date')

    #date asse x non tutte ma solo alcune 
    ax1.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    
for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()


## New cases vs behaviour

In [ ]:
df['new_cases_smoothed'].max()

In [ ]:
import matplotlib.ticker as ticker
def millions_formatter(x, pos):
    if x >= 1e6:
        return f'{x*1e-6:.1f}M'  # or .0f if we don't want decimals
    elif x >= 1e3:
        return f'{x*1e-3:.0f}K'
    else:
        return f'{int(x)}'


df['new_cases_smoothed'] = df['new_cases_smoothed']/df['new_cases_smoothed'].max()
behcases = df[['country', 'pct_mask', 'survey_date', 'total_cases', 'new_cases_smoothed']].copy()
behcases['survey_date'] = pd.to_datetime(behcases['survey_date'])

lista_paesi =behcases['country'].unique()  
fig, axs = plt.subplots(12, 3, figsize=(18, 50), sharex=False)
axs = axs.flatten() 

for i, paese in enumerate(lista_paesi):
    df_paese = behcases[behcases['country'] == paese].copy()
    df_paese.set_index('survey_date', inplace=True)     # datetime index
    # 7days mean
    pct_mask_smooth = df_paese['pct_mask'].rolling('7D', min_periods=1).mean()
    df_smooth = df_paese.copy()
    df_smooth['pct_mask'] = pct_mask_smooth
    # index for plot
    df_smooth.reset_index(inplace=True)

    ax1 = axs[i]
    ax1.plot(df_smooth['survey_date'], df_smooth['pct_mask'], color='blue', label='Mask Wearing Behaviour (%)')
    ax1.set_ylabel('Mask Wearing Behaviour (%)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    ax2 = ax1.twinx()
    ax2.plot(df_smooth['survey_date'], df_smooth['new_cases_smoothed'], color='red', label='New Cases')
    ax2.set_ylabel('New Cases', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(millions_formatter)) # add "M"

    ax1.set_title(paese)
    ax1.set_xlabel('Survey Date')

    #date asse x non tutte ma solo alcune 
    ax1.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    #ax1.tick_params(axis='x', labelrotation=90)

for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()

In [ ]:
# Format axes y
def millions_formatter(x, pos):
    if x >= 1e6:
        return f'{x*1e-6:.1f}M'
    elif x >= 1e3:
        return f'{x*1e-3:.0f}K'
    else:
        return f'{int(x)}'

behcases = df[['country', 'pct_mask', 'survey_date', 'total_cases', 'new_cases_smoothed']].copy()
behcases['survey_date'] = pd.to_datetime(behcases['survey_date'])
behcases = behcases[behcases['country'] == 'New Zealand']

behcases.set_index('survey_date', inplace=True)
pct_mask_smooth = behcases['pct_mask'].rolling('7D', min_periods=1).mean()
behcases['pct_mask_smooth'] = pct_mask_smooth
behcases.reset_index(inplace=True)


fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(behcases['survey_date'], behcases['pct_mask_smooth']*100, color='blue', label='Mask Wearing Behaviour (%)')
ax1.set_ylabel('Mask Wearing Behaviour (%)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xlabel('Survey Date')
ax1.set_title('New Zealand')

ax2 = ax1.twinx()
ax2.plot(behcases['survey_date'], behcases['new_cases_smoothed'], color='red', label='New Cases')
ax2.set_ylabel('New Cases', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(millions_formatter))

ax1.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))


plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator

# format y axis
def millions_formatter(x, pos):
    if x >= 1e6:
        return f'{x*1e-6:.1f}M'
    elif x >= 1e3:
        return f'{x*1e-3:.0f}K'
    else:
        return f'{int(x)}'

# bhecases
behcases = df[['country', 'pct_mask', 'survey_date', 'total_cases', 'new_cases_smoothed']].copy()
behcases['survey_date'] = pd.to_datetime(behcases['survey_date'])
behcases = behcases[behcases['country'] == 'New Zealand']
behcases.set_index('survey_date', inplace=True)
behcases['pct_mask_smooth'] = behcases['pct_mask'].rolling('7D', min_periods=1).mean()
behcases.reset_index(inplace=True)

# face covering
ds = pd.read_csv('face-covering-policies-covid.csv')
ds_nz = ds[ds['Entity'] == 'New Zealand'].copy()
ds_nz['Day'] = pd.to_datetime(ds_nz['Day'])
ds_nz.rename(columns={'Day': 'survey_date', 'Facial Coverings': 'facial_covering'}, inplace=True)

behcases = pd.merge_asof(
    behcases.sort_values('survey_date'),
    ds_nz[['survey_date', 'facial_covering']].sort_values('survey_date'),
    on='survey_date',
    direction='backward'
)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True)  # ideale per A4 orizzontale

# --- LEFT plot: mask wearing + facial covering ---
color_mask = 'blue'
color_policy = 'green'

ax1.plot(behcases['survey_date'], behcases['pct_mask_smooth']*100, color=color_mask, label='Mask Wearing Behaviour (%)')
ax1.set_ylabel('Mask Wearing Behaviour (%)', color=color_mask, fontsize=14)
ax1.set_xlabel('Survey Date', fontsize=14)
ax1.set_title('Behaviour and Facial Covering Policy (New Zealand)', fontsize=14)
ax1.tick_params(axis='y', labelcolor=color_mask, labelsize=12)
ax1.tick_params(axis='x', labelsize=12)

# Secondary axis for facial covering
ax1b = ax1.twinx()
ax1b.plot(behcases['survey_date'], behcases['facial_covering'], color=color_policy, linestyle='--', label='Facial Covering Policy (0–4)')
ax1b.set_ylabel('Facial Covering Policy (0–4)', color=color_policy, fontsize=14)
ax1b.tick_params(axis='y', labelcolor=color_policy, labelsize=12)
ax1b.set_ylim(0, 4)
ax1b.yaxis.set_major_locator(MultipleLocator(1))

# --- RIGHT graph: new cases smoothed ---
color_cases = 'red'
ax2.plot(behcases['survey_date'], behcases['new_cases_smoothed'], color=color_cases, label='New Cases')
ax2.set_ylabel('New Cases', color=color_cases, fontsize=14)
ax2.set_xlabel('Survey Date', fontsize=14)
ax2.set_title('New COVID-19 Cases (New Zealand)', fontsize=14)
ax2.tick_params(axis='y', labelcolor=color_cases, labelsize=12)
ax2.tick_params(axis='x', labelsize=12)
#ax2.yaxis.set_major_formatter(ticker.FuncFormatter(millions_formatter))
ax2.legend(loc='upper left', fontsize=12)


for ax in [ax1, ax2]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()


### MLR

In [ ]:

b = df[['pct_mask', 'new_deaths_smoothed', 'new_cases_smoothed']].copy().dropna() 
print(b[['new_deaths_smoothed', 'new_cases_smoothed']].corr()     )            # independent?

X_vif = add_constant(b[['new_deaths_smoothed', 'new_cases_smoothed']])
vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print("\nVariance Inflation Factor (VIF): ")
print(vif_data)

X = b[['new_deaths_smoothed', 'new_cases_smoothed']]
y = b['pct_mask']

model = LinearRegression().fit(X, y)

print(" \n Coefficienti -beta:", model.coef_)
print("Intercetta -alpha:", model.intercept_)
print("R^2:", model.score(X, y))

In [ ]:
# This additional analysis allows to disentangle the pooled estimate and isolates the within-country association 
# by removing the removing the country-specific average as well as the temporal within-country and corss-country correlations

# Panel-indexed DataFrame (sorted by country)
b = df[['country', 'survey_date', 'pct_mask', 'new_deaths_smoothed', 'new_cases_smoothed' ]].copy()
b = b.dropna()
b['survey_date'] = pd.to_datetime(b['survey_date'])
b = b.set_index(['country', 'survey_date'])

y = b['pct_mask']
X = b[[ 'new_deaths_smoothed', 'new_cases_smoothed' ]]

# Country fixed-effects ("within") estimator; entity_effects=True implements the demeaning
model_fe = PanelOLS(y, X, entity_effects=True)

# Cluster-robust SEs by country: robust to within-country autocorrelation of arbitrary form
fe_clustered = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(fe_clustered.summary)

# Driscoll-Kraay SEs: robust to within-country autocorrelation AND cross-country contemporaneous correlation (e.g. shared global shocks/news events). 
# Doesn't assume that countries are independent and includes heteroskedasticity
fe_dk = model_fe.fit(cov_type='kernel', kernel='bartlett', bandwidth=12)
print(fe_dk.summary)

#### granger causality

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'new_cases_smoothed']].dropna(), variables =['pct_mask', 'new_cases_smoothed']) 


In [ ]:
corr1=calculate_correlations("pct_mask","new_deaths_smoothed")
corr2=calculate_correlations("pct_mask", "new_cases_smoothed")
boxplot2(corr1,corr2, 'New Deaths', 'New Cases', 'Epidemiological', 'NCND', 'forestgreen')

## Behaviour vs CHI

In [ ]:
df['Containment health index'] = df['Containment health index']/df['Containment health index'].max()

behchi = df[['country', 'pct_mask', 'survey_date', 'Containment health index']].copy()
behchi['survey_date'] = pd.to_datetime(behchi['survey_date'])


lista_paesi =behchi['country'].unique()  
fig, axs = plt.subplots(12, 3, figsize=(18, 50), sharex=False)
axs = axs.flatten() 

for i, paese in enumerate(lista_paesi):
    df_paese = behchi[behchi['country'] == paese].copy()
    df_paese.set_index('survey_date', inplace=True)     
   
    df_smooth = df_paese[['pct_mask']].rolling('7D', min_periods=1).mean()

    
    df_smooth.reset_index(inplace=True)
   

    ax1 = axs[i]
    ax1.plot(df_smooth['survey_date'], df_smooth['pct_mask'], color='blue', label='Mask Wearing Behaviour (%)')
    ax1.set_ylabel('Mask Wearing Behaviour (%)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.set_ylim(0, 1)     


    ax2 = ax1.twinx()
    ax2.plot(df_smooth['survey_date'], df_paese['Containment health index'], color='red', label='Total Cases')
    ax2.set_ylabel('Containment health index', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(millions_formatter)) 
    ax2.set_ylim(0, 1)

    ax1.set_title(paese)
    ax1.set_xlabel('Survey Date')

    ax1.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()

In [ ]:
behchi = df[['country', 'pct_mask', 'survey_date', 'Containment health index']].copy()
behchi['survey_date'] = pd.to_datetime(behchi['survey_date'])
behchi = behchi[behchi['country'].isin(['Japan', 'Thailand'])]


lista_paesi = ['Japan', 'Thailand']
fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharex=False)
axs = axs.flatten() 

for i, paese in enumerate(lista_paesi):
    df_paese = behchi[behchi['country'] == paese].copy()
    df_paese.set_index('survey_date', inplace=True)     
    
    df_smooth = df_paese[['pct_mask']].rolling('7D', min_periods=1).mean()

    # cambia indice per il plot
    df_smooth.reset_index(inplace=True)
   

    ax1 = axs[i]
    ax1.plot(df_smooth['survey_date'], df_smooth['pct_mask'], color='blue', label='Mask Wearing Behaviour (%)')
    ax1.set_ylabel('Mask Wearing Behaviour (%)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.set_ylim(0, 1)     


    ax2 = ax1.twinx()
    ax2.plot(df_smooth['survey_date'], df_paese['Containment health index'], color='red', label='Total Cases')
    ax2.set_ylabel('Containment health index', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(millions_formatter)) 
    ax2.set_ylim(0, 1)

    ax1.set_title(paese)
    ax1.set_xlabel('Survey Date')

    
    ax1.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax1.tick_params(axis='x', labelrotation=45)


for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()

### Boxplot

In [ ]:
cor = calculate_correlations("pct_mask", "Containment health index")
df1 = cor.copy()
df1['dataset'] = 'CHI'
combined_df = df1

color = 'peru'

plt.figure(figsize=(3, 6.5))
ax = sns.boxplot(
    x='dataset',
    y='correlation',
    data=combined_df,
    width=0.3, 
    showfliers=False,
    boxprops=dict(facecolor='white', edgecolor=color, linewidth=1),
    whiskerprops=dict(color=color, linewidth=0.7),
    capprops=dict(color=color, linewidth=0.7),
    medianprops=dict(color=color, linewidth=1)
)

add_outliers(ax, df1, 0, color)
ax.set_xticklabels(['CHI'], fontsize=14)

plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
plt.ylim(-1, 1)
plt.title('Interventions', fontsize=14, pad=15, color=color)
plt.ylabel('Correlations', fontsize=14)
plt.xlabel('', fontsize=14)
plt.grid(axis='y', linestyle=':', alpha=0.3)
plt.xlim(-0.5, 0.5)
ax.tick_params(axis='x', labelrotation=15)

plt.tight_layout()
plt.show()


In [ ]:
print(shannon('Malaysia', 'pct_mask'))
print(shannon('Malaysia', 'Containment health index'))

In [ ]:
## MLR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

b = df[['country','pct_mask', 'Containment health index']].copy().dropna()
b = b[b['country'] == 'Malaysia']

X = b[['Containment health index']]
y = b['pct_mask']

X = sm.add_constant(X)  # aggiunge la colonna 'const'

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:

b = df[['country','pct_mask', 'Containment health index']].copy().dropna() 


X = b[['Containment health index']]
y = b['pct_mask']

model = LinearRegression().fit(X, y)

print("Coefficienti -beta:", model.coef_)
print("Intercetta -alpha:", model.intercept_)
print("R^2:", model.score(X, y))

In [ ]:
# This additional analysis allows to disentangle the pooled estimate and isolates the within-country association 
# by removing the removing the country-specific average as well as the temporal within-country and corss-country correlations

# Panel-indexed DataFrame (sorted by country)
b = df[['country', 'survey_date', 'pct_mask',  'Containment health index']].copy()
b = b.dropna()
b['survey_date'] = pd.to_datetime(b['survey_date'])
b = b.set_index(['country', 'survey_date'])

y = b['pct_mask']
X = b[[ 'Containment health index']]

# Country fixed-effects ("within") estimator; entity_effects=True implements the demeaning
model_fe = PanelOLS(y, X, entity_effects=True)

# Cluster-robust SEs by country: robust to within-country autocorrelation of arbitrary form
fe_clustered = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(fe_clustered.summary)

# Driscoll-Kraay SEs: robust to within-country autocorrelation AND cross-country contemporaneous correlation (e.g. shared global shocks/news events). 
# Doesn't assume that countries are independent and includes heteroskedasticity
fe_dk = model_fe.fit(cov_type='kernel', kernel='bartlett', bandwidth=12)
print(fe_dk.summary)

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'Containment health index']].dropna(), variables =['pct_mask', 'Containment health index']) 

From here on, behaviours are not anymore the output variables, but I look for the interdependencies of other variables.

## CHI  vs BEL; AWA; TRU

In [ ]:
corr1=calculate_correlations("Containment health index","pct_worried_catch_covid")
corr2=calculate_correlations("Containment health index", "pct_belief_masking_effective")
boxplot2(corr1,corr2, 'Bel1', 'Bel2', 'CHI vs Beliefs', 'CHIBel', "black")

In [ ]:
print(shannon('Austria', 'Containment health index'))
print(shannon('Germany', 'Containment health index'))
print(shannon('Austria', 'pct_worried_catch_covid'))
print(shannon('Germany', 'pct_worried_catch_covid'))

In [ ]:
b = df[['country','pct_worried_catch_covid', 'Containment health index', 'pct_belief_masking_effective']].copy().dropna()
# Regression: CHI → pct_worried_catch_covid
y1 = b['pct_worried_catch_covid']
X = sm.add_constant(b['Containment health index'])
model1 = sm.OLS(y1, X).fit()
print(model1.summary())

In [ ]:
# Regression: CHI → pct_belief_masking_effective
y2 = b['pct_belief_masking_effective']
model2 = sm.OLS(y2, X).fit()
print(model2.summary())

In [ ]:
y1 = b['Containment health index']

X = sm.add_constant(b['pct_belief_masking_effective'])
model1 = sm.OLS(y1, X).fit()
print(model1.summary())

In [ ]:
grangers_causation_matrix(
    df[['Containment health index', 'pct_worried_catch_covid']].dropna(), 
    variables=['Containment health index', 'pct_worried_catch_covid']
)

In [ ]:
grangers_causation_matrix(
    df[['Containment health index', 'pct_belief_masking_effective']].dropna(), 
    variables=['Containment health index', 'pct_belief_masking_effective']
)

In [ ]:
corr1=calculate_correlations("Containment health index","pct_received_news_local_health")
corr2=calculate_correlations("Containment health index","pct_received_news_experts")
corr3=calculate_correlations("Containment health index","pct_received_news_who")
corr4=calculate_correlations("Containment health index","pct_received_news_govt_health")
corr5=calculate_correlations("Containment health index","pct_received_news_politicians")
corr6=calculate_correlations("Containment health index","pct_received_news_journalists")
corr7=calculate_correlations("Containment health index","pct_received_news_friends")
corr=[corr1, corr2, corr3, corr4, corr5, corr6, corr7]
names=['Awa1', 'Awa2','Awa3','Awa4','Awa5','Awa6','Awa7']
boxplot7(corr, names, 'CHI vs Awareness', 'CHIAwa',"black")

In [ ]:
plot_aree(corr6, 'Country correlation of CHI and Awareness6 (Journalists)', 'CHI-awa-analisi6')

In [ ]:
b = df[['country', 'Containment health index', 
        'pct_received_news_local_health', 'pct_received_news_experts', 
        'pct_received_news_who', 'pct_received_news_govt_health', 
        'pct_received_news_politicians', 'pct_received_news_journalists', 
        'pct_received_news_friends']].copy().dropna()

# Prima regressione: CHI → awareness variables
y1 = b['Containment health index']
X = sm.add_constant(b[['pct_received_news_local_health', 
                       'pct_received_news_experts', 
                       'pct_received_news_who', 
                       'pct_received_news_govt_health', 
                       'pct_received_news_politicians', 
                       'pct_received_news_journalists', 
                       'pct_received_news_friends']])
model1 = sm.OLS(y1, X).fit()
print(model1.summary())





In [ ]:
cort1=calculate_correlations("Containment health index","pct_trust_covid_info_local_health")
cort2=calculate_correlations("Containment health index","pct_trust_covid_info_experts")
cort3=calculate_correlations("Containment health index","pct_trust_covid_info_who")
cort4=calculate_correlations("Containment health index","pct_trust_covid_info_govt_health")
cort5=calculate_correlations("Containment health index","pct_trust_covid_info_politicians")
cort6=calculate_correlations("Containment health index","pct_trust_covid_info_journalists")
cort7=calculate_correlations("Containment health index","pct_trust_covid_info_friends")
cort=[cort1, cort2, cort3, cort4, cort5, cort6, cort7]
names=['Tru1', 'Tru2','Tru3','Tru4','Tru5','Tru6','Tru7']
boxplot7(cort, names, 'CHI vs Trust', 'CHITru', "black")

## New cases vs beliefs

In [ ]:
corr1=calculate_correlations("new_cases_smoothed","pct_worried_catch_covid")
corr2=calculate_correlations("new_cases_smoothed", "pct_belief_masking_effective")
boxplot2(corr1,corr2, 'Bel1', 'Bel2', 'New Cases vs Beliefs', 'CasiBel',"black")

In [ ]:
grangers_causation_matrix(
    df[['new_cases_smoothed', 'pct_worried_catch_covid']].dropna(), 
    variables=['new_cases_smoothed', 'pct_worried_catch_covid']
)

In [ ]:
grangers_causation_matrix(
    df[['new_cases_smoothed', 'pct_belief_masking_effective']].dropna(), 
    variables=['new_cases_smoothed', 'pct_belief_masking_effective']
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

b = df[['country','new_cases_smoothed', 'pct_belief_masking_effective','pct_worried_catch_covid' ]].copy().dropna()

X = b[['pct_belief_masking_effective','pct_worried_catch_covid']]
y = b['new_cases_smoothed']

X = sm.add_constant(X)  # aggiunge la colonna 'const'

model = sm.OLS(y, X).fit()

print(model.summary())

## cases death VS chi

In [ ]:
corr1=calculate_correlations("Containment health index", "new_cases_smoothed")
corr2=calculate_correlations( "Containment health index", "new_deaths_smoothed")
boxplot2(corr1,corr2, 'New Cases', 'New Deaths', 'CHI vs Epidemiological Indicators', 'CHIcasideaths',"black")

In [ ]:
grangers_causation_matrix(
    df[['Containment health index', 'new_cases_smoothed']].dropna(), 
    variables=['Containment health index', 'new_cases_smoothed']
)

In [ ]:
grangers_causation_matrix(
    df[['Containment health index', 'new_deaths_smoothed']].dropna(), 
    variables=['Containment health index', 'new_deaths_smoothed']
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

b = df[['country','Containment health index','new_cases_smoothed', 'new_deaths_smoothed' ]].copy().dropna()

X = b[['new_cases_smoothed', 'new_deaths_smoothed']]
y = b['Containment health index']

X = sm.add_constant(X)  # add 'const'

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
X = b[['Containment health index']]
y = b['new_cases_smoothed']  

X = sm.add_constant(X) 

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
X = b[['Containment health index']]
y = b['new_deaths_smoothed']   

X = sm.add_constant(X)  

model = sm.OLS(y, X).fit()

print(model.summary())

## Beliefs vs Awareness

In [ ]:
corr1=calculate_correlations("pct_worried_catch_covid","pct_received_news_local_health")
corr2=calculate_correlations("pct_worried_catch_covid","pct_received_news_experts")
corr3=calculate_correlations("pct_worried_catch_covid","pct_received_news_who")
corr4=calculate_correlations("pct_worried_catch_covid","pct_received_news_govt_health")
corr5=calculate_correlations("pct_worried_catch_covid","pct_received_news_politicians")
corr6=calculate_correlations("pct_worried_catch_covid","pct_received_news_journalists")
corr7=calculate_correlations("pct_worried_catch_covid","pct_received_news_friends")
corr=[corr1, corr2, corr3, corr4, corr5, corr6, corr7]
names=['Awa1', 'Awa2','Awa3','Awa4','Awa5','Awa6','Awa7']
boxplot7(corr, names, 'Belief "Worried Catch COVID" vs Awareness', 'CHIAwa',"black")


In [ ]:
corr1=calculate_correlations("pct_belief_masking_effective","pct_received_news_local_health")
corr2=calculate_correlations("pct_belief_masking_effective","pct_received_news_experts")
corr3=calculate_correlations("pct_belief_masking_effective","pct_received_news_who")
corr4=calculate_correlations("pct_belief_masking_effective","pct_received_news_govt_health")
corr5=calculate_correlations("pct_belief_masking_effective","pct_received_news_politicians")
corr6=calculate_correlations("pct_belief_masking_effective","pct_received_news_journalists")
corr7=calculate_correlations("pct_belief_masking_effective","pct_received_news_friends")
corr=[corr1, corr2, corr3, corr4, corr5, corr6, corr7]
names=['Awa1', 'Awa2','Awa3','Awa4','Awa5','Awa6','Awa7']
boxplot7(corr, names, 'Belief "Mask Effective" vs Awareness', 'CHIAwa',"black")

## Add analysis on Stringency Index
To make a sort of "sensitivity analysis" on the policies, about whether and how much the health-related policies matter: make a comparison by using the Stringency Index, which is like the CHI but without the health indicators.

In [ ]:
# Filtered list of countries
countries = df['country'].unique().tolist()

# Database with SI; make sure that the keys in both dg and df match
dg = pd.read_csv('owid-covid-data.csv')  # fetch SI data from the OWID dataset
dg.loc[dg['location'] == "Czechia", 'location'] = "Czech Republic"
dg= dg[dg['location'].isin(countries)]
dg['country']= dg['location']
dg['date'] = pd.to_datetime(dg['date'])
dg['date'] = dg['date'].dt.date
dg= dg[(dg['date'] >= date(2021, 5, 21)) & (dg['date'] <= date(2022, 6, 25))]
df["date"] = pd.to_datetime(df["survey_date"])
dg["date"] = pd.to_datetime(dg["date"])

# drop elements from dg with no correspondence in df (a few dates are missing for some countries
key = ["country", "date"]
dg_filtered = dg.merge(df[key].drop_duplicates(), on=key, how="inner")

# Merge in one database
df["stringency_index"] = dg_filtered['stringency_index'].values
df['stringency_index'] = df['stringency_index']/df['stringency_index'].max()

### Boxplot

In [ ]:
cor = calculate_correlations("pct_mask", "stringency_index")
df1 = cor.copy()
df1['dataset'] = 'stringency_index'
combined_df = df1

color = 'peru'

plt.figure(figsize=(3, 6.5))
ax = sns.boxplot(
    x='dataset',
    y='correlation',
    data=combined_df,
    width=0.3, 
    showfliers=False,
    boxprops=dict(facecolor='white', edgecolor=color, linewidth=1),
    whiskerprops=dict(color=color, linewidth=0.7),
    capprops=dict(color=color, linewidth=0.7),
    medianprops=dict(color=color, linewidth=1)
)

add_outliers(ax, df1, 0, color)
ax.set_xticklabels(['Stringency Index'], fontsize=14)

plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
plt.ylim(-1, 1)
plt.title('Interventions', fontsize=14, pad=15, color=color)
plt.ylabel('Correlations', fontsize=14)
plt.xlabel('', fontsize=14)
plt.grid(axis='y', linestyle=':', alpha=0.3)
plt.xlim(-0.5, 0.5)
ax.tick_params(axis='x', labelrotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# This additional analysis allows to disentangle the pooled estimate and isolates the within-country association 
# by removing the removing the country-specific average as well as the temporal within-country and corss-country correlations

# Panel-indexed DataFrame (sorted by country)
b = df[['country', 'survey_date', 'pct_mask',  'stringency_index']].copy()
b = b.dropna()
b['survey_date'] = pd.to_datetime(b['survey_date'])
b = b.set_index(['country', 'survey_date'])

y = b['pct_mask']
X = b[[ 'stringency_index']]

# Country fixed-effects ("within") estimator; entity_effects=True implements the demeaning
model_fe = PanelOLS(y, X, entity_effects=True)

# Cluster-robust SEs by country: robust to within-country autocorrelation of arbitrary form
fe_clustered = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(fe_clustered.summary)

# Driscoll-Kraay SEs: robust to within-country autocorrelation AND cross-country contemporaneous correlation (e.g. shared global shocks/news events). 
# Doesn't assume that countries are independent and includes heteroskedasticity
fe_dk = model_fe.fit(cov_type='kernel', kernel='bartlett', bandwidth=12)
print(fe_dk.summary)

In [ ]:
grangers_causation_matrix(df[['pct_mask', 'stringency_index']].dropna(), variables =['pct_mask', 'stringency_index']) 